In [3]:
from pathlib import Path
import gc
import re
import unicodedata
from difflib import SequenceMatcher

import torch
import torchaudio
import whisperx

from praatio import textgrid
from praatio.data_classes.interval_tier import IntervalTier


# ============================================================
# CONFIG
# ============================================================

PROJECT_ROOT = Path("/workspace/storyfx")

EXCERPT_ID = "excerpt_001"

RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / f"{EXCERPT_ID}"
    / "raw"
)

AUDIO_PATH = (
    RAW_DIR
    / f"{EXCERPT_ID}.wav"
)

TRANSCRIPT_PATH = (
    RAW_DIR
    / f"{EXCERPT_ID}.txt"
)

OUTPUT_TEXTGRID = (
    RAW_DIR
    / f"{EXCERPT_ID}.TextGrid"
)


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

COMPUTE_TYPE = (
    "float16"
    if DEVICE == "cuda"
    else "float32"
)

WHISPER_MODEL = "large-v3"

BATCH_SIZE = 8

TARGET_SEGMENT_S = 30.0

# Search this far around the 30 s target.
BOUNDARY_TOLERANCE_S = 5.0

# Prefer pauses at least this large.
MIN_PREFERRED_GAP_S = 0.15


# ============================================================
# TEXT HELPERS
# ============================================================

def normalize_token(text):
    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    text = (
        text
        .replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9']+",
        "",
        text,
    )

    return text


def source_tokens_with_spans(text):
    """
    Returns tokens from the original transcript while preserving
    character offsets so exact capitalization/punctuation can later
    be recovered.
    """

    pattern = re.compile(
        r"[A-Za-z0-9]+(?:['’][A-Za-z0-9]+)?"
    )

    tokens = []

    for match in pattern.finditer(text):
        raw = match.group(0)

        tokens.append({
            "raw": raw,
            "norm": normalize_token(raw),
            "start_char": match.start(),
            "end_char": match.end(),
        })

    return tokens


# ============================================================
# LOAD SOURCE TRANSCRIPT
# ============================================================

source_text = TRANSCRIPT_PATH.read_text(
    encoding="utf-8"
).strip()

source_tokens = source_tokens_with_spans(
    source_text
)

print(
    f"Source transcript words: "
    f"{len(source_tokens)}"
)


# ============================================================
# AUDIO DURATION
# ============================================================

audio_info = torchaudio.info(
    str(AUDIO_PATH)
)

audio_duration = (
    audio_info.num_frames
    / audio_info.sample_rate
)

print(
    f"Audio duration: "
    f"{audio_duration:.2f}s"
)


# ============================================================
# WHISPERX ASR + SILERO VAD
# ============================================================

print(
    "\nLoading WhisperX ASR model..."
)

model = whisperx.load_model(
    WHISPER_MODEL,
    DEVICE,
    compute_type=COMPUTE_TYPE,
    language="en",
    vad_method="silero",
    vad_options={
        "chunk_size": 30,
        "vad_onset": 0.5,
        "vad_offset": 0.363,
    },
)

audio = whisperx.load_audio(
    str(AUDIO_PATH)
)

print(
    "\nTranscribing full audio..."
)

result = model.transcribe(
    audio,
    batch_size=BATCH_SIZE,
    chunk_size=30,
)

print(
    f"Whisper segments: "
    f"{len(result['segments'])}"
)


# Free ASR model before alignment.
del model

gc.collect()

if DEVICE == "cuda":
    torch.cuda.empty_cache()


# ============================================================
# WHISPERX WORD ALIGNMENT
# ============================================================

print(
    "\nLoading alignment model..."
)

align_model, metadata = (
    whisperx.load_align_model(
        language_code="en",
        device=DEVICE,
    )
)

print(
    "Running WhisperX alignment..."
)

aligned = whisperx.align(
    result["segments"],
    align_model,
    metadata,
    audio,
    DEVICE,
    return_char_alignments=False,
)


del align_model

gc.collect()

if DEVICE == "cuda":
    torch.cuda.empty_cache()


# ============================================================
# FLATTEN ALIGNED WORDS
# ============================================================

whisper_words = []

for segment in aligned["segments"]:

    for word in segment.get(
        "words",
        []
    ):
        if (
            word.get("start") is None
            or word.get("end") is None
        ):
            continue

        raw_word = word.get(
            "word",
            ""
        ).strip()

        norm = normalize_token(
            raw_word
        )

        if not norm:
            continue

        whisper_words.append({
            "raw": raw_word,
            "norm": norm,
            "start": float(
                word["start"]
            ),
            "end": float(
                word["end"]
            ),
        })


print(
    f"Aligned Whisper words: "
    f"{len(whisper_words)}"
)


# ============================================================
# MAP WHISPER WORDS TO SOURCE TRANSCRIPT
# ============================================================

whisper_norm = [
    w["norm"]
    for w in whisper_words
]

source_norm = [
    w["norm"]
    for w in source_tokens
]


matcher = SequenceMatcher(
    None,
    whisper_norm,
    source_norm,
    autojunk=False,
)


# whisper index -> source index
whisper_to_source = {}


for block in matcher.get_matching_blocks():

    for offset in range(
        block.size
    ):
        whisper_index = (
            block.a
            + offset
        )

        source_index = (
            block.b
            + offset
        )

        whisper_to_source[
            whisper_index
        ] = source_index


matched_count = len(
    whisper_to_source
)

print(
    f"Matched words: "
    f"{matched_count}/"
    f"{len(whisper_words)}"
)

print(
    f"Match rate: "
    f"{100 * matched_count / max(1, len(whisper_words)):.1f}%"
)


# ============================================================
# FILL SMALL ALIGNMENT GAPS
# ============================================================

def interpolate_source_indices(
    whisper_words,
    mapping,
    source_count,
):
    """
    Fill unmatched Whisper-token positions by interpolating
    between the surrounding Whisper→source matches.

    This is only used for small ASR mismatches. Exact matches
    from SequenceMatcher remain unchanged.
    """

    mapped = sorted(
        mapping.items()
    )

    if not mapped:
        raise RuntimeError(
            "Whisper transcript could not be "
            "mapped to source transcript."
        )

    full_mapping = dict(
        mapping
    )

    # ----------------------------------------
    # Before first mapped token
    # ----------------------------------------

    first_w, first_s = mapped[0]

    for w in range(
        first_w - 1,
        -1,
        -1,
    ):
        estimate = (
            first_s
            - (first_w - w)
        )

        if estimate >= 0:
            full_mapping[w] = estimate

    # ----------------------------------------
    # Between mapped tokens
    # ----------------------------------------

    for (
        left_w,
        left_s,
    ), (
        right_w,
        right_s,
    ) in zip(
        mapped,
        mapped[1:],
    ):

        w_gap = (
            right_w
            - left_w
        )

        s_gap = (
            right_s
            - left_s
        )

        if w_gap <= 1:
            continue

        for w in range(
            left_w + 1,
            right_w,
        ):
            ratio = (
                (w - left_w)
                / w_gap
            )

            estimate = round(
                left_s
                + ratio * s_gap
            )

            estimate = max(
                left_s,
                min(
                    estimate,
                    right_s,
                ),
            )

            full_mapping[w] = estimate

    # ----------------------------------------
    # After last mapped token
    # ----------------------------------------

    last_w, last_s = mapped[-1]

    for w in range(
        last_w + 1,
        len(whisper_words),
    ):
        estimate = (
            last_s
            + (w - last_w)
        )

        if estimate < source_count:
            full_mapping[w] = estimate

    return full_mapping


whisper_to_source = (
    interpolate_source_indices(
        whisper_words,
        whisper_to_source,
        len(source_tokens),
    )
)


# ============================================================
# FIND ~30 SECOND BOUNDARIES
# ============================================================

def choose_boundary(
    words,
    start_index,
    target_time,
    tolerance_s=5.0,
):
    """
    Find the best word boundary near target_time.

    Preference:
    1. boundary inside tolerance
    2. larger pause
    3. closer to target time
    """

    lower = (
        target_time
        - tolerance_s
    )

    upper = (
        target_time
        + tolerance_s
    )

    candidates = []

    for i in range(
        start_index,
        len(words) - 1,
    ):
        current_end = (
            words[i]["end"]
        )

        next_start = (
            words[i + 1]["start"]
        )

        if current_end < lower:
            continue

        if current_end > upper:
            break

        gap = max(
            0.0,
            next_start
            - current_end,
        )

        distance = abs(
            current_end
            - target_time
        )

        preferred_gap = (
            gap
            >= MIN_PREFERRED_GAP_S
        )

        candidates.append({
            "word_index": i,
            "boundary": (
                current_end
                + next_start
            ) / 2.0,
            "gap": gap,
            "distance": distance,
            "preferred": preferred_gap,
        })

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: (
            not x["preferred"],
            -x["gap"],
            x["distance"],
        )
    )

    return candidates[0]


# ============================================================
# BUILD SEGMENTS
# ============================================================

segments = []

start_whisper_index = 0
segment_start_time = 0.0


while (
    start_whisper_index
    < len(whisper_words)
):

    target_time = (
        segment_start_time
        + TARGET_SEGMENT_S
    )

    # Final segment.
    if (
        target_time
        >= audio_duration
    ):
        end_whisper_index = (
            len(whisper_words)
            - 1
        )

        segment_end_time = (
            audio_duration
        )

    else:

        chosen = choose_boundary(
            whisper_words,
            start_whisper_index,
            target_time,
            BOUNDARY_TOLERANCE_S,
        )

        if chosen is None:

            # Fallback: closest word end
            # to target.
            candidates = [
                (
                    abs(
                        w["end"]
                        - target_time
                    ),
                    i,
                )
                for i, w
                in enumerate(
                    whisper_words
                )
                if (
                    i
                    >= start_whisper_index
                )
            ]

            _, end_whisper_index = min(
                candidates,
                key=lambda x: x[0],
            )

            segment_end_time = (
                whisper_words[
                    end_whisper_index
                ]["end"]
            )

            print(
                f"[FALLBACK] "
                f"Boundary near "
                f"{target_time:.2f}s "
                f"-> "
                f"{segment_end_time:.2f}s"
            )

        else:

            end_whisper_index = (
                chosen[
                    "word_index"
                ]
            )

            segment_end_time = (
                chosen[
                    "boundary"
                ]
            )

            print(
                f"[BOUNDARY] "
                f"target="
                f"{target_time:.2f}s "
                f"selected="
                f"{segment_end_time:.2f}s "
                f"gap="
                f"{chosen['gap']:.3f}s"
            )


    # --------------------------------------------------------
    # Map this Whisper segment onto original transcript
    # --------------------------------------------------------

    mapped_source_indices = [
        whisper_to_source[i]
        for i in range(
            start_whisper_index,
            end_whisper_index + 1,
        )
        if i in whisper_to_source
    ]

    if not mapped_source_indices:
        raise RuntimeError(
            "Could not map segment back "
            "to source transcript."
        )

    source_start_index = min(
        mapped_source_indices
    )

    source_end_index = max(
        mapped_source_indices
    )


    # --------------------------------------------------------
    # Recover exact text from original transcript
    # --------------------------------------------------------

    if segments:
        # Start exactly after previous source section.
        previous_source_end = (
            segments[-1][
                "source_end_index"
            ]
        )

        source_start_index = (
            previous_source_end
            + 1
        )

    if (
        end_whisper_index
        == len(whisper_words) - 1
    ):
        # Last segment gets remainder of source transcript.
        source_end_index = (
            len(source_tokens) - 1
        )


    start_char = (
        source_tokens[
            source_start_index
        ]["start_char"]
    )

    if (
        source_end_index
        + 1
        < len(source_tokens)
    ):
        # Include punctuation/whitespace up to next word.
        end_char = (
            source_tokens[
                source_end_index + 1
            ]["start_char"]
        )
    else:
        end_char = len(
            source_text
        )


    segment_text = (
        source_text[
            start_char:end_char
        ]
        .strip()
    )


    segments.append({
        "xmin": (
            segment_start_time
        ),
        "xmax": (
            segment_end_time
        ),
        "text": (
            segment_text
        ),
        "source_start_index": (
            source_start_index
        ),
        "source_end_index": (
            source_end_index
        ),
    })


    print(
        f"  "
        f"{segment_start_time:.2f}"
        f" -> "
        f"{segment_end_time:.2f}"
    )

    print(
        f"  "
        f"{segment_text[:100]}"
        f"..."
    )

    print()


    start_whisper_index = (
        end_whisper_index
        + 1
    )

    segment_start_time = (
        segment_end_time
    )


# ============================================================
# CLEAN INTERVAL EDGES
# ============================================================

# Ensure first interval begins at exactly 0.
segments[0]["xmin"] = 0.0

# Ensure contiguous intervals.
for i in range(
    1,
    len(segments)
):
    segments[i]["xmin"] = (
        segments[i - 1]["xmax"]
    )

# Ensure final interval reaches exact audio end.
segments[-1]["xmax"] = (
    audio_duration
)


# ============================================================
# TEXTGRID ENTRIES
# ============================================================

entries = []

for segment in segments:

    entries.append(
        (
            round(
                segment["xmin"],
                3,
            ),
            round(
                segment["xmax"],
                3,
            ),
            segment["text"],
        )
    )


# ============================================================
# SAVE TEXTGRID
# ============================================================

tier = IntervalTier(
    name="segments",
    entries=entries,
    minT=0.0,
    maxT=audio_duration,
)

tg = textgrid.Textgrid()

tg.addTier(
    tier
)

tg.save(
    str(
        OUTPUT_TEXTGRID
    ),
    format="long_textgrid",
    includeBlankSpaces=True,
)


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n=============================="
)

print(
    f"Created "
    f"{len(entries)} segments"
)

print(
    "TextGrid saved to:"
)

print(
    OUTPUT_TEXTGRID
)

print(
    "==============================\n"
)

for i, (
    xmin,
    xmax,
    text,
) in enumerate(entries):

    print(
        f"{i:02d}: "
        f"{xmin:.2f} "
        f"-> "
        f"{xmax:.2f} "
        f"({xmax - xmin:.2f}s)"
    )

    print(
        text[:120]
    )

    print()

Source transcript words: 971


/tmp/ipykernel_32617/2060165508.py:148: UserWarning: torchaudio._backend.utils.info has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  audio_info = torchaudio.info(
/workspace/whisperx_env/lib/python3.12/site-packages/torchaudio/_backend/ffmpeg.py:20: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  s 

Audio duration: 388.60s

Loading WhisperX ASR model...
2026-10-03 21:27:37 - whisperx.vads.silero - INFO - Performing voice activity detection using Silero...


Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master



Transcribing full audio...
Whisper segments: 14

Loading alignment model...
Running WhisperX alignment...
Aligned Whisper words: 962
Matched words: 946/962
Match rate: 98.3%
[BOUNDARY] target=30.00s selected=31.05s gap=0.501s
  0.00 -> 31.05
  For three centuries, Harrow’s Peak had stood as a testament to the ancestral hubris of the ancient B...

[BOUNDARY] target=61.05s selected=56.85s gap=0.700s
  31.05 -> 56.85
  and in the whispered folklore of the nearby village, which spoke of the "Hollow Ones" who roamed the...

[BOUNDARY] target=86.85s selected=91.98s gap=1.040s
  56.85 -> 91.98
  The Eastwoods did not own the manor; they were its mere custodians, bound by an unspoken, familial p...

[BOUNDARY] target=121.98s selected=120.24s gap=0.981s
  91.98 -> 120.24
  That night, Corbin barely heard the chimes against the scream of wind-charged rain that had been rag...

[BOUNDARY] target=150.24s selected=152.01s gap=0.820s
  120.24 -> 152.01
  In forty years he had spent his time at Harr